In [0]:
%sql
CREATE TABLE IF NOT EXISTS migration.silver.customers (
    customer_id BIGINT,
    customer_name STRING,
    email STRING,
    city STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _source_batch_id STRING,
    _source_run_id STRING,
    _source_ingestion_timestamp TIMESTAMP,
    _processed_timestamp TIMESTAMP
)
USING DELTA;

## Find the latest version of every customer

In [0]:
%sql
WITH ranked_customers AS (

    SELECT
        customer_id,
        customer_name,
        email,
        city,
        created_at,
        updated_at,
        _batch_id,
        _run_id,
        _ingestion_timestamp,

        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY updated_at DESC, _ingestion_timestamp DESC
        ) AS rn

    FROM migration.bronze.customers
)

SELECT
    customer_id,
    customer_name,
    email,
    city,
    created_at,
    updated_at,
    _batch_id,
    _run_id,
    _ingestion_timestamp
FROM ranked_customers
WHERE rn = 1;

-- SELECT * from ranked_customers;

In [0]:
%sql
WITH latest AS (

    SELECT
        customer_id,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY updated_at DESC, _ingestion_timestamp DESC
        ) AS rn

    FROM migration.bronze.customers
)

SELECT
    customer_id,
    COUNT(*) AS cnt
FROM latest
WHERE rn = 1
GROUP BY customer_id
HAVING COUNT(*) > 1;